# Script del pipeline de datos

A continuación el código del pipeline de procesamiento de datos como se solicita en `instrucciones.md`:

```
Código que: 
(1) carga el documento, 
(2) divide el texto en chunks (20+ chunks),
(3) genera embeddings para todos los chunks y
(4) guarda embeddings + chunks en algún almacenamiento (archivo, base de datos o en memoria). 

Debe ser ejecutable.
```

### Inicialización del del pipeline

In [1]:
# Librerias
from openai import OpenAI
import chromadb
from dotenv import load_dotenv
from os import getenv
from src.log_gen import log
from src.pricing_estimates import guardar_metricas_embedding
from datetime import datetime
import tiktoken
from time import perf_counter

load_dotenv()

# Variables de entorno
EMBEDDINGS_MODEL=str(getenv('EMBEDDINGS_MODEL'))
CHROMADB_PATH=str(getenv('CHROMADB_PATH'))
COLLECTION_NAME=str(getenv('COLLECTION_NAME'))

print('Librerias y variables de entorno cargadas')

Librerias y variables de entorno cargadas


In [2]:
# Instanciar clientes

# ChromaDB
db_client = chromadb.PersistentClient(path=CHROMADB_PATH)

# Cliente de OpenAI
embeddings_client = OpenAI()

print('Clientes instanciados.')


Clientes instanciados.


**Nota: Se debe indicar la misma variable de control aquí que la que se escogió en `00 data-pipeline.ipynb` para que agente consulte por la colección correspondiente.**

In [ ]:
'''
Variables de control
'''
# Descomentar solo una.
estrategia_chunking = [
    'SEMANTICA',
    #'FIXED_SIZE',
    # ...
]

if len(estrategia_chunking) != 1:
    raise Exception('Debe utilizarse solo una estrategia a la vez.')

estrategia_chunking = estrategia_chunking[0]

# Crear colección si no existe
final_collection_name = f'{COLLECTION_NAME}_{estrategia_chunking}'
collection = db_client.get_or_create_collection(
    name=final_collection_name,
    metadata={"hnsw:space": "cosine"}
)

print(f'Estrategia de chunking seleccionada: {estrategia_chunking}')

Estrategia de chunking seleccionada: SEMANTICA


In [4]:
embeddings_encoder = tiktoken.encoding_for_model(EMBEDDINGS_MODEL)

def crear_embedding(texto):
    timestamp = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

    # Estimar tokens
    tokens_prompt = len(embeddings_encoder.encode(texto))

    # Llamar api de OpenAI Embeddings
    
    t1 = perf_counter()
    response = embeddings_client.embeddings.create(
        model=EMBEDDINGS_MODEL,
        input=texto
    )
    t2 = perf_counter()
    response_time = (t2 - t1) * 1000

    # Log y guardado de métricas
    log(f'Respuesta de API Embeddings recibida | Tiempo: {response_time:.0f} ms | Tokens - prompt: {tokens_prompt}')
    guardar_metricas_embedding(timestamp, EMBEDDINGS_MODEL, tokens_prompt, response_time)

    # devolver embedding
    return response.data[0].embedding

**1. Carga del documento**

In [5]:
# Carga del documento
with open('data/faq_document.md', mode='r', encoding='utf8') as file:
    document = file.read()

**2. Divide el texto en chunks (20+ chunks)**

In [6]:


# Subdividimos el documento segun la estrategia escogida:
if estrategia_chunking == 'SEMANTICA':
    from src.chunking_semantico import semantic_chunking

    chunks = semantic_chunking(document)
    print('Chunks generados.')
    

elif estrategia_chunking == 'FIXED_SIZE':
    from src.chunking_fixed_size import fixed_size_chunking

    """
    # Calcular para obtener al menos 20 chunks del documento
    MIN_CHUNKS = 20
    total_palabras = len(document.split(' '))
    print(total_palabras / MIN_CHUNKS) # --> 62.25 palabras por chunk.
    """

    chunk_size = 50 # En n° de palabras
    overlap = 15 # En n° de palabras

    chunks = fixed_size_chunking(document, chunk_size, overlap)
    print('Chunks generados.')

Chunks generados.


**3. Genera embeddings para todos los chunks**

In [7]:
# Generación de embeddings via OpenAI
for chunk in chunks:
    #print("-"*30)
    #print("Generando Embeddings para el fragmento:")
    #print(chunk['text'])

    chunk['embedding'] = crear_embedding(chunk['text'])

print('Embeddings generados correctamente.')

[2026-03-01 16:56:24 INFO] Respuesta de API Embeddings recibida | Tiempo: 681 ms | Tokens - prompt: 149
[2026-03-01 16:56:24 INFO] Métricas escritas: ['2026-03-01T16:56:23', 149, 0, 149, '681', '0.00000298USD']
[2026-03-01 16:56:24 INFO] Respuesta de API Embeddings recibida | Tiempo: 266 ms | Tokens - prompt: 87
[2026-03-01 16:56:24 INFO] Métricas escritas: ['2026-03-01T16:56:24', 87, 0, 87, '266', '0.00000174USD']
[2026-03-01 16:56:24 INFO] Respuesta de API Embeddings recibida | Tiempo: 258 ms | Tokens - prompt: 152
[2026-03-01 16:56:24 INFO] Métricas escritas: ['2026-03-01T16:56:24', 152, 0, 152, '258', '0.00000304USD']
[2026-03-01 16:56:24 INFO] Respuesta de API Embeddings recibida | Tiempo: 238 ms | Tokens - prompt: 127
[2026-03-01 16:56:24 INFO] Métricas escritas: ['2026-03-01T16:56:24', 127, 0, 127, '238', '0.00000254USD']
[2026-03-01 16:56:25 INFO] Respuesta de API Embeddings recibida | Tiempo: 264 ms | Tokens - prompt: 190
[2026-03-01 16:56:25 INFO] Métricas escritas: ['2026-03

**4. Guarda embeddings + chunks en algún almacenamiento (archivo, base de datos o en memoria).**

In [8]:
collection.add(
    ids=[chunk["id"] for chunk in chunks],
    embeddings=[chunk["embedding"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[chunk["metadata"] for chunk in chunks]
)

print(f'Chunks con embeddings añadidos a colección "{final_collection_name}" en ChromaDB.')
print(f'Colección "{final_collection_name}" tiene {collection.count()} documentos.')

Chunks con embeddings añadidos a colección "documentos_SEMANTICA" en ChromaDB.
Colección "documentos_SEMANTICA" tiene 24 documentos.
